# Churn Intelligence — V3: Robustness Testing

Teste de robustez do modelo de churn no estilo **empresarial** (não FGSM acadêmico): perturbação de features, missing-data attacks, distribution shift e detecção de out-of-distribution. Consolidado em `robustness_report`.

Modela a pergunta: *se `tenure: 14→15`, `tickets: 5→4`, `usage: 120→125` já muda a probabilidade de churn de 0.82 para 0.39, a fronteira de decisão é confiável?*

## Setup — modelo sobre features.csv

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from src.robustness import (feature_perturbation, fragility_score, missing_data_sweep,
                            covariate_shift, psi, OODDetector, robustness_report)

df = pd.read_csv('../data/processed/features.csv')
target = 'Churned' if 'Churned' in df.columns else 'churn'
num = df.select_dtypes(include=[np.number]).drop(columns=[target], errors='ignore')
y = df[target].astype(int).to_numpy()
split = int(0.7*len(num))
X_tr, X_te = num.iloc[:split], num.iloc[split:]
y_tr, y_te = y[:split], y[split:]
model = RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print('acurácia teste:', (model.predict(X_te) == y_te).mean())


## 1. Feature perturbation — fragilidade da fronteira

In [ ]:
key = [c for c in ['recency_days','frequency','avg_session_duration','intensity',
                    'Customer_Service_Calls','Days_Since_Last_Purchase'] if c in X_te.columns][:5]
for f in key:
    print(f'
{f}:')
    for row in feature_perturbation(model, X_te, f, deltas=[-2,-1,1,2]):
        print(f"  delta {row['delta']:+.1f} -> flip {row['decision_flip_rate']:.1%}, "
              f"|Δprob| {row['mean_abs_prob_change']:.3f}")
print('
fragility_score (±1):', fragility_score(model, X_te, key))


## 2. Missing-data attacks — de quais features o modelo mais depende

In [ ]:
for m in missing_data_sweep(model, X_te, key):
    print(f"{m['feature']:<28} flip {m['decision_flip_rate']:.1%}  |Δprob| {m['mean_abs_prob_change']:.3f}")


## 3. Distribution shift — queda de acurácia sob covariate shift

In [ ]:
f0 = key[0]
for r in covariate_shift(model, X_te, y_te, f0, scales=(1.0,1.25,1.5,2.0)):
    print(f"scale {r['scale']:<4} -> acc {r['accuracy']:.3f} (drop {r['accuracy_drop']:+.3f}), "
          f"drift prob média {r['mean_prob_drift']:+.3f}")
print('PSI treino vs shift x2:', round(psi(X_tr[f0].to_numpy(), (X_te[f0]*2).to_numpy()), 3))


## 4. OOD detection

In [ ]:
det = OODDetector().fit(X_tr)
print('flag rate no teste (in-dist):', det.flag_rate(X_te))
X_ood = X_te.copy(); X_ood[f0] = X_ood[f0]*50 + 1e4
print('flag rate sob input OOD      :', det.flag_rate(X_ood))


## 5. Relatório consolidado

In [ ]:
rep = robustness_report(model, X_tr, X_te, y_te, key_features=key)
print(rep['rendered'])


## Nota

O `robustness_report` alimenta o *robustness gate* do **Argus** (`ml-platform/adversarial-evaluation/`) via o `ModelSecurityReport` do **ThemisAI** — um modelo de churn com fronteira muito manipulável ou muito sensível a shift não é promovido a produção.